In [1]:
# Use this cell to directly run ONLY the from_pretrained and load previous saved LoRA weights
!pip install --upgrade pip
!pip install -q accelerate diffusers transformers peft torch torchvision
!pip install -q ipywidgets jupyterlab dataclass_wizard

  Obtaining dependency information for pip from https://files.pythonhosted.org/packages/c9/bc/b7db44f5f39f9d0494071bddae6880eb645970366d0a200022a1a93d57f5/pip-25.0.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.6 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 23.2.1
    Uninstalling pip-23.2.1:
      Successfully uninstalled pip-23.2.1


In [2]:
!pip install label-studio-sdk

INFO: pip is looking at multiple versions of pydantic[email] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 160.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 531.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 218.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 549.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 559.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.2
    Uninstalling numpy-2.1.2:
      Successfully uninstalled numpy-2.1.2
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.18
    Uninstalling pydantic-1.10.18:
      Successfully uninstalled pydantic-1.10.18
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the

In [14]:
# Define the URL where Label Studio is accessible and the API key for your user account
LABEL_STUDIO_URL = 'http://label-studio-route-zhang-aws.apps.ai-dev01.kni.syseng.devcluster.openshift.com'
# API key is available at the Account & Settings > Access Tokens page in Label Studio UI
API_KEY = 'd79c5a9c19db16194b6c1ae5d2724d852711328d'

PROJECT_TITLE='jupiternotebook'

# Import the SDK and the client module
from label_studio_sdk.client import LabelStudio

# Connect to the Label Studio API and check the connection
ls = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=API_KEY)

In [15]:
from label_studio_sdk.label_interface import LabelInterface
from label_studio_sdk.label_interface.create import choices
# Define labeling interface

label_config = """
<View>
    <Text name="input_text" value="$text"/>
  <TextArea name="generated_text"  toName="input_text"/>
</View>
    """
projects = ls.projects.list()
project = None
for found in projects:
    if found.title == PROJECT_TITLE:
        project = found
        print("found", project)
        break
if project is None:
    # Create a project with the specified title and labeling configuration
    print("create new project: ", PROJECT_TITLE)
    project = ls.projects.create(
        title=PROJECT_TITLE,
        label_config=label_config
    )

found id=4 title='jupiternotebook' description='' label_config='<View>\n    <Text name="input_text" value="$text"/>\n  <TextArea name="generated_text"  toName="input_text"/>\n</View>' expert_instruction='' show_instruction=False show_skip_button=True enable_empty_annotation=True show_annotation_history=False organization=1 prompts=None color='#FFFFFF' maximum_annotations=1 is_published=False model_version='' is_draft=False created_by=UserSimple(id=1, first_name='', last_name='', email='jianrzha@redhat.com', avatar=None) created_at=datetime.datetime(2025, 3, 4, 20, 12, 26, 835838, tzinfo=TzInfo(UTC)) min_annotations_to_start_training=0 start_training_on_annotation_update=False show_collab_predictions=True num_tasks_with_annotations=0 task_number=0 useful_annotation_number=0 ground_truth_number=0 skipped_annotations_number=0 total_annotations_number=0 total_predictions_number=0 sampling='Sequential sampling' show_ground_truth_first=False show_overlap_first=False overlap_cohort_percentage

In [13]:
print(project)

id=4 title='jupiternotebook' description='' label_config='<View>\n    <Text name="input_text" value="$text"/>\n  <TextArea name="generated_text"  toName="input_text"/>\n</View>' expert_instruction='' show_instruction=False show_skip_button=True enable_empty_annotation=True show_annotation_history=False reveal_preannotations_interactively=False show_collab_predictions=True maximum_annotations=1 color='#FFFFFF' control_weights={'generated_text': {'overall': 1.0, 'type': 'TextArea', 'labels': {}}} organization=1 is_published=False model_version='' is_draft=False created_by={'id': 1, 'first_name': '', 'last_name': '', 'email': 'jianrzha@redhat.com', 'avatar': None} created_at='2025-03-04T20:12:26.835838Z' min_annotations_to_start_training=0 start_training_on_annotation_update=False num_tasks_with_annotations=None task_number=None useful_annotation_number=None ground_truth_number=None skipped_annotations_number=None total_annotations_number=None total_predictions_number=None sampling='Seque

In [17]:
ls.projects.import_tasks(
    id=project.id,
    request=[
        {"text": "Where is the town of Sudbury?"},
        {"text": "What is the total population of Sudbury?"},
        {"text": "What is percentage of asian in Sudbury?"},
    ]
)

ProjectsImportTasksResponse(task_count=3, annotation_count=0, predictions_count=None, duration=0.02298116683959961, file_upload_ids=[], could_be_tasks_list=False, found_formats=[], data_columns=[], prediction_count=0)

In [18]:
data = ls.projects.exports.as_json(project.id)
print(data)


[{'id': 5, 'annotations': [{'id': 1, 'completed_by': 1, 'result': [{'id': 'DINhXEObpC', 'type': 'textarea', 'value': {'text': ['The town of Sudbury is located in the Metrowest area of Massachusetts.']}, 'origin': 'manual', 'to_name': 'input_text', 'from_name': 'generated_text'}], 'was_cancelled': False, 'ground_truth': False, 'created_at': '2025-03-05T15:06:20.616234Z', 'updated_at': '2025-03-05T15:06:20.616242Z', 'draft_created_at': None, 'lead_time': 53.757, 'prediction': {}, 'result_count': 1, 'unique_id': '52d48bf4-e188-43bd-9403-e8af3d5813aa', 'import_id': None, 'last_action': None, 'bulk_created': False, 'task': 5, 'project': 4, 'updated_by': 1, 'parent_prediction': None, 'parent_annotation': None, 'last_created_by': None}], 'drafts': [], 'predictions': [], 'data': {'text': 'Where is the town of Sudbury?'}, 'meta': {}, 'created_at': '2025-03-05T15:04:54.822747Z', 'updated_at': '2025-03-05T15:06:20.644102Z', 'inner_id': 1, 'total_annotations': 1, 'cancelled_annotations': 0, 'total